# Generative training on LSDJ data

### Setup

In [1]:
%%capture
!git clone https://github.com/exilefaker/pe-lsdj.git

%cd pe-lsdj
%pip install -e .

In [2]:
import sys
sys.path.append('/content/pe-lsdj/src')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
DATA_FOLDER = 'LSDJ/data/'
TRAIN_FOLDER = 'train/'
VALIDATE_FOLDER = 'validate/'
CHECKPOINT_FOLDER = 'LSDJ/checkpoints'

DATA_PATH = DRIVE_PATH + DATA_FOLDER
TRAIN_PATH = DATA_PATH + TRAIN_FOLDER
VALIDATE_PATH = DATA_PATH + VALIDATE_FOLDER
CHECKPOINT_PATH = DRIVE_PATH + CHECKPOINT_FOLDER

Mounted at /content/drive


### Tokenize song data

In [3]:
import glob
from pe_lsdj import SongFile
from pe_lsdj.training import load_songs

song_files = sorted(glob.glob(TRAIN_PATH + "*.lsdsng"))
validate_files = sorted(glob.glob(VALIDATE_PATH + "*.lsdsng"))

songs: list[SongFile] = load_songs(song_files)
validate_songs: list[SongFile] = load_songs(validate_files)

### Train model

In [4]:
from pe_lsdj.models import LSDJTransformer
import jax.random as jr

key = jr.PRNGKey(33)
model_key, train_key = jr.split(key)
model = LSDJTransformer(
    model_key,
    d_model=512, # Try a bigger model
    fx_dim=128,
    instr_entity_dim=256,
    num_blocks=8,
    noise_sd=0.05,
    dropout_p=0.1,
)

In [ ]:
from pe_lsdj.training import train

model, opt_state = train(
    model, 
    songs,
    validation_songs=validate_songs,
    key=train_key,
    batch_size=4,
    crop_len=128,
    checkpoint_path=CHECKPOINT_PATH,
    resume_from_checkpoint=True,
    # weight_decay=0.05,
    # label_smoothing=0.05,
    # num_steps=1000, # Best weights occur around step ~500
    transpose_range=2, # Data augmentations
    swap_pulse=True,
)


Resumed from /content/drive/MyDrive/LSDJ/checkpoints/2026-03-15_03:50:25_995293 [ v8.2   512 ]/step_000350.eqx (step 350)
step   350 | PULSE+FORMAT+RAWK+RISKY | loss 5.8262 | val 6.5630
step   400 | XTRA+CHINA+MEGAMANX+DECISIV2 | loss 4.9323 | val 6.4461
step   450 | ODE+SHINK0R+RISKY+XTRA | loss 4.3339 | val 6.5853
step   500 | RISKY+CRSHH+MEGAMANX+SHINK0R | loss 5.0833 | val 6.4153
step   550 | PULSE+FORMAT+RAWK+CHINA | loss 3.6155 | val 6.4921
step   600 | CHORAL+MEGAMAN+XTRA+GTRZ | loss 2.8925 | val 6.4506
step   650 | PULSE+GTRZ+HIZ+CRSHH | loss 1.4142 | val 6.4195
step   700 | CHINA+HIZ+RAWK+EQUUS | loss 2.9900 | val 6.4586
step   750 | HIZ+CRSHH+RISKY+PULSE | loss 2.6536 | val 6.7083
step   800 | MEGAMAN+FORMAT+EQUUS+XTRA | loss 3.4334 | val 6.9904
step   850 | RAWK+PULSE+DECISIV2+MEGAMANX | loss 2.3537 | val 6.6454
step   900 | MEGAMAN+CRSHH+FORMAT+CHINA | loss 2.8563 | val 6.6798
step   950 | MEGAMAN+RISKY+CHINA+B1TMAP | loss 2.7212 | val 7.0191
step  1000 | ODE+RISKY+SHINK0R+